In [15]:
import os
import requests
from typing import TypedDict
from typing import Annotated
from IPython.display import Image, display
from dotenv import load_dotenv

from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
from langgraph.prebuilt import ToolNode, tools_condition

from langchain_community.agent_toolkits import FileManagementToolkit
from langchain_experimental.tools import PythonREPLTool
from langchain_community.utilities import GoogleSerperAPIWrapper

In [16]:
load_dotenv(override=True)


True

In [17]:
"""
Q. explain what FileManagementToolkit toolkit does?
FileManagementToolkit is a LangChain agent toolkit that gives an LLM (agent) the ability to interact with the local file system in a controlled way in a specific directory.
It bundles together several file-related tools so an agent can read, write, list, move, and delete files while reasoning about them.
"""
def get_file_tools():
    toolkit = FileManagementToolkit(root_dir="codes")
    return toolkit.get_tools()

file_tools = get_file_tools()
print(f'file_tools: {file_tools}')

file_tools: [CopyFileTool(root_dir='codes'), DeleteFileTool(root_dir='codes'), FileSearchTool(root_dir='codes'), MoveFileTool(root_dir='codes'), ReadFileTool(root_dir='codes'), WriteFileTool(root_dir='codes'), ListDirectoryTool(root_dir='codes')]


In [20]:
# step 1
llm = ChatOpenAI(model="gpt-4o-mini")


# step 2
tools = get_file_tools()
llm_with_tools = llm.bind_tools(tools)

# step 3, create a graph with LangGraph to use tools
class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

# Create Edges
graph_builder.add_edge(START, "chatbot")

graph_builder.add_conditional_edges( "chatbot", tools_condition, "tools")

# Any time a tool is called, we return to the chatbot to decide the next step
graph_builder.add_edge("tools", "chatbot")

# Compile the Graph
graph = graph_builder.compile()

# draw and display the graph
# display(Image(graph.get_graph().draw_mermaid_png()))

# invoke the graph with a message that requires tool use
# user_input = "Create a file called test.txt with the content 'Hello, World!'"
user_input = "Create a python script file called python1.py that can add two numbers."

result = graph.invoke({"messages": [{"role": "user", "content": user_input}]})
print(f'result: {result}')

result: {'messages': [HumanMessage(content='Create a python script file called python1.py that can add two numbers.', additional_kwargs={}, response_metadata={}, id='fa84fd4d-7f2d-4b5b-9817-f1cdc7e6b007'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_FRb4UshGu4fvfd7POxUyFF12', 'function': {'arguments': '{"file_path":"python1.py","text":"def add_numbers(num1, num2):\\n    return num1 + num2\\n\\n# Example usage\\nif __name__ == \'__main__\':\\n    number1 = float(input(\'Enter first number: \'))\\n    number2 = float(input(\'Enter second number: \'))\\n    sum_result = add_numbers(number1, number2)\\n    print(f\'The sum of {number1} and {number2} is: {sum_result}\')\\n","append":false}', 'name': 'write_file'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 324, 'total_tokens': 450, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'reject